# 02 — GPX : parsing, normalisation et rééchantillonnage

Ce notebook construit la première brique du pipeline.

Objectif :

```text
fichier GPX
    ↓
points GPS bruts
    ↓
distance cumulée
    ↓
rééchantillonnage spatial régulier
    ↓
route normalisée tous les 10 m
```

Le rééchantillonnage est nécessaire parce que les points d'un GPX réel sont rarement espacés régulièrement.
Les agents et détecteurs doivent travailler sur une base spatiale commune.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import gpxpy
import matplotlib.pyplot as plt

## 1. Distance géodésique approximée

Pour la V1, on calcule la distance entre deux points GPS avec la formule de Haversine.

Cela suffit pour construire une distance cumulée le long de l'itinéraire.

In [ ]:
EARTH_RADIUS_M = 6_371_000.0

def haversine_m(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return EARTH_RADIUS_M * c

## 2. Lecture d'un GPX

Le loader retourne un DataFrame avec :

- `lat`
- `lon`
- `elev_m`
- `distance_m`

Si plusieurs segments sont présents, ils sont concaténés dans l'ordre.

In [ ]:
def load_gpx(path):
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        gpx = gpxpy.parse(f)

    rows = []

    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                rows.append({
                    "lat": point.latitude,
                    "lon": point.longitude,
                    "elev_m": point.elevation,
                })

    if len(rows) < 2:
        raise ValueError("Le GPX doit contenir au moins 2 points.")

    df = pd.DataFrame(rows)

    distances = [0.0]
    for i in range(1, len(df)):
        d = haversine_m(
            df.loc[i - 1, "lat"],
            df.loc[i - 1, "lon"],
            df.loc[i, "lat"],
            df.loc[i, "lon"],
        )
        distances.append(distances[-1] + d)

    df["distance_m"] = distances
    return df

## 3. Route synthétique de secours

Tant qu'on n'a pas placé de vrai `route_test.gpx` dans `data/input/`, ce bloc permet de tester tout le pipeline.

La route contient volontairement :
- une montée ;
- une portion raide ;
- plusieurs changements de direction ressemblant à des lacets ;
- une descente.

In [ ]:
def make_synthetic_route(n=500, spacing_m=10.0):
    distance = np.arange(n) * spacing_m

    # Coordonnées locales simulées autour d'un point GPS
    lat0 = 45.0
    lon0 = 6.0

    # Route en plan avec oscillations plus fortes au milieu
    x = distance.copy()
    y = np.zeros_like(distance, dtype=float)

    mask = (distance >= 1500) & (distance <= 3000)
    y[mask] = 80 * np.sin((distance[mask] - 1500) / 120)

    # Conversion grossière mètres -> degrés pour visualisation/prototype
    lat = lat0 + y / 111_320
    lon = lon0 + x / (111_320 * np.cos(np.radians(lat0)))

    # Profil altimétrique
    elev = np.zeros_like(distance, dtype=float)
    elev += 200
    elev += np.where(distance < 1200, 0.03 * distance, 36)
    elev += np.where((distance >= 1200) & (distance < 2200), 0.12 * (distance - 1200), 0)
    elev += np.where(distance >= 2200, 120, 0)
    elev -= np.where(distance >= 3200, 0.08 * (distance - 3200), 0)

    return pd.DataFrame({
        "lat": lat,
        "lon": lon,
        "elev_m": elev,
        "distance_m": distance,
    })

## 4. Rééchantillonnage spatial

On interpole la route sur un pas régulier, par défaut 10 m.

In [ ]:
def resample_route(df, spacing_m=10.0):
    if "distance_m" not in df.columns:
        raise ValueError("La route doit contenir distance_m.")

    max_d = float(df["distance_m"].iloc[-1])
    new_d = np.arange(0, max_d + spacing_m, spacing_m)

    out = pd.DataFrame({"distance_m": new_d})

    for col in ["lat", "lon", "elev_m"]:
        values = df[col].astype(float).to_numpy()
        valid = ~np.isnan(values)

        if valid.sum() < 2:
            out[col] = np.nan
        else:
            out[col] = np.interp(
                new_d,
                df["distance_m"].to_numpy()[valid],
                values[valid]
            )

    return out

## 5. Charger un vrai GPX si présent

Déposer un fichier ici :

```text
data/input/route_test.gpx
```

Sinon le notebook utilisera automatiquement la route synthétique.

In [ ]:
gpx_path = Path("data/input/route_test.gpx")

if gpx_path.exists():
    route_raw = load_gpx(gpx_path)
    print("GPX réel chargé :", gpx_path)
else:
    route_raw = make_synthetic_route()
    print("Aucun GPX trouvé : utilisation d'une route synthétique.")

route = resample_route(route_raw, spacing_m=10.0)

print(route.head())
print("Nombre de points :", len(route))
print("Longueur :", round(route["distance_m"].iloc[-1], 1), "m")

In [ ]:
fig, ax = plt.subplots()
ax.plot(route["distance_m"], route["elev_m"])
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Altitude (m)")
ax.set_title("Profil altimétrique")
plt.show()

## 6. Sortie attendue pour la suite

L'objet `route` est le format commun utilisé dans les prochains notebooks :

```text
distance_m | lat | lon | elev_m
```

Pour la V1, tous les détecteurs travailleront sur cette représentation normalisée.